# Konu 1 — Veriyi kodla işlemek: kelime bulutu

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

## Isınma — geçen dönemin iki hatası

**Tamir 1.** Öğrencinin puanını (85) yazmalı. Hata veriyor. Neden?

In [ ]:
def puani_getir(ogrenci):
    return ogrenci[1]

print(puani_getir({"ad": "Deniz", "puan": 85}))

`ogrenci` bir sözlük. Sözlükte sıra numarası yok; Python `1` diye bir **anahtar** arıyor.
Sözlük **anahtarla** açılır:

In [ ]:
def puani_getir(ogrenci):
    return ogrenci["puan"]

print(puani_getir({"ad": "Deniz", "puan": 85}))

**Tamir 2.** Toplamı (60) yazmalı. Hata yok, ama sonuç yanlış. Neden?

In [ ]:
def topla(sayilar):
    toplam = 0
    for sayi in sayilar:
        toplam = toplam + sayi
        return toplam

print(topla([10, 20, 30]))

`return` döngünün içinde: fonksiyon ilk turda biter. Bir girinti sola alalım:

In [ ]:
def topla(sayilar):
    toplam = 0
    for sayi in sayilar:
        toplam = toplam + sayi
    return toplam

print(topla([10, 20, 30]))

## Adım 1 — Dosyayı aç

Bir kafenin müşterilerinin yazdığı 30 yorum. Müşteriler bu kafeyi nasıl anlatıyor?
Defter, kendi durduğu klasörde çalışır; `veri/...` bu klasörün içindeki `veri` demek.

In [ ]:
with open("veri/kafe-yorumlari.txt", encoding="utf-8") as dosya:
    metin = dosya.read()

print(len(metin))
print(metin[:150])

## Adım 2 — Kelimelere ayır

Altıncı ve yedinci kelimeye bak: noktalı, virgüllü, büyük harfli.

In [ ]:
kelimeler = metin.split()
print(len(kelimeler))
print(kelimeler[:8])

### Sorun 1 — Türkçe küçültme

`.lower()` Türkçe **İ** ve **I** harflerini doğru küçültmüyor:

In [ ]:
print("İnternet".lower() == "internet")
print("Işık".lower())

Önce bu iki harfi kendimiz çeviriyoruz, **sonra** küçültüyoruz. Her satır tek iş:

In [ ]:
temiz = metin.replace("I", "ı")
temiz = temiz.replace("İ", "i")
temiz = temiz.lower()

### Sorun 2 — Noktalama

Noktalama işaretlerini boşluğa çevirip yeniden bölüyoruz.

In [ ]:
for isaret in ".,;:!?'()-":
    temiz = temiz.replace(isaret, " ")

kelimeler = temiz.split()
print(kelimeler[:8])

## Adım 3 — Say

Sözlük: anahtar kelime, değer kaç kez geçtiği.

In [ ]:
sayac = {}
for kelime in kelimeler:
    if kelime in sayac:
        sayac[kelime] = sayac[kelime] + 1
    else:
        sayac[kelime] = 1

print(len(sayac))
print(sayac["sessiz"])

## Adım 4 — En sık geçenler

`key=sayac.get` "değerine göre sırala" demek. `sayac.get`'ten sonra parantez **yok**.

In [ ]:
sirali = sorted(sayac, key=sayac.get, reverse=True)
for kelime in sirali[:20]:
    print(sayac[kelime], kelime)

En üstte **için, var, bir, ama, çok**. Bunlar kafe hakkında bir şey söylemiyor.

## Adım 5 — Durak kelimeleri ele

Durak kelimeler hazır bir dosyada, her satırda bir kelime.

In [ ]:
with open("veri/turkce-durak-kelimeler.txt", encoding="utf-8") as dosya:
    durak_metni = dosya.read()

durak_kelimeler = durak_metni.split()
print(len(durak_kelimeler))
print(durak_kelimeler[:10])

Adım 3'teki sayaç, başına **tek koşul** eklenmiş hâliyle:

In [ ]:
sayac = {}
for kelime in kelimeler:
    if kelime not in durak_kelimeler and len(kelime) > 2:
        if kelime in sayac:
            sayac[kelime] = sayac[kelime] + 1
        else:
            sayac[kelime] = 1

In [ ]:
sirali = sorted(sayac, key=sayac.get, reverse=True)
for kelime in sirali[:10]:
    print(sayac[kelime], kelime)

Müşteriler kafeyi kahvesiyle değil, **sessiz bir çalışma yeri** olarak anlatıyor.

## Adım 6 — Kelime bulutu

Buluta kendi sayacımızı veriyoruz. Kelimenin boyu, sayaçtaki değeri.

In [ ]:
from wordcloud import WordCloud

bulut = WordCloud(width=1200, height=800, background_color="white")
bulut.generate_from_frequencies(sayac)
bulut.to_image()

Hücreyi bir kez daha çalıştır: konum ve renk değişir, **boyut değişmez**.
Yalnızca boyut veri taşır.

Tasarımla oyna: `background_color="black"` yap, parantezin içine `colormap="magma"` ya da
`max_words=25` ekle, yeniden çalıştır.

## Adım 7 — Bulutun göremediği: ekler ve bağlam

`kahve`, `kahvesi`, `kahvenin` ayrı sayılıyor. İçinde "kahve" geçenleri toplayalım:

In [ ]:
aranan = "kahve"
bulunanlar = []
for kelime in kelimeler:
    if aranan in kelime:
        bulunanlar.append(kelime)

print(len(bulunanlar))
print(bulunanlar)

Kahve 6'ya çıktı, prizi geçti. `"kahve"` yerine `"bahçe"`, `"sessiz"` yazıp yeniden çalıştır.

Sayı **kaç kez** geçtiğini söyler, **nasıl** geçtiğini söylemez. Kahveden söz eden
yorumları okuyalım. `.splitlines()` metni satırlara böler; her satır bir yorum.

In [ ]:
for satir in temiz.splitlines():
    if "kahve" in satir:
        print(satir)

Kahveyi doğrudan öven kaç yorum var? Şimdi ışık:

In [ ]:
for satir in temiz.splitlines():
    if "ışık" in satir:
        print(satir)

Bulutta neredeyse görünmeyen bir kelime, somut bir tasarım sorunu çıkardı.
Sayma **nereye bakacağını** söyler; bulguyu **bağlamda okuyarak** bulursun.

## Adım 8 — Aynı sonuç, çubuk grafik

`plt.bar` iki liste ister: adlar ve boylar. Adlar hazır (`sirali`), boyları topluyoruz.

In [ ]:
import matplotlib.pyplot as plt

etiketler = sirali[:10]
degerler = []
for kelime in etiketler:
    degerler.append(sayac[kelime])

In [ ]:
plt.bar(etiketler, degerler, color="#4a6fa5")
plt.title("Müşteri yorumlarında en sık 10 kelime")
plt.ylabel("Kaç kez geçti")
plt.xticks(rotation=45)
plt.show()

Bulut "neyi anlatıyor"u sezdirir; grafik "ne kadar önde"yi **ölçer**.

## Bonus — Kafenin kendi gönderileri

Adım 1'deki dosya adını `veri/kafe-gonderileri.txt` yap, hücreleri yeniden sırayla
çalıştır. Adım 3'teki `sayac["sessiz"]` satırı ne diyor? Kafe kendini neyle anlatıyor?